In [1]:
import json
import urllib.request
import time
from pathlib import Path
from IPython.display import display, Markdown

# =========================================================
# CONFIG
# =========================================================
OLLAMA_URL   = "http://localhost:11434/api/generate"
OLLAMA_MODEL = "mistral"

# ── 9 zones océaniques ──
ZONES = {
    "Océan Pacifique Sud": {
        "description": "Océan Pacifique Sud, Polynésie, Mélanésie, Micronésie, côtes australiennes, îles Galápagos",
        "escales": ["Île de Crespo", "Archipel des Pomotou", "Archipel des nouvelles Hébrides", "La Pouasie", "Cap Wessel", "Mer de Timor"]
    },
    "Océan Indien": {
        "description": "Océan Indien, côtes indiennes, Sri Lanka, île Keeling, golfe du Bengale, archipel des Maldives",
        "escales": ["Île Keeling", "Golfe du Bengale"]
    },
    "Mer Rouge": {
        "description": "Mer Rouge, golfe d'Aden, détroit de Babel-Mandeb, côtes arabiques et africaines orientales",
        "escales": ["Isthme de Suez"]
    },
    "Méditerranée": {
        "description": "Mer Méditerranée, mer Égée, mer Adriatique, détroit de Gibraltar, archipel grec",
        "escales": ["L'Archipel grec", "Méditerranée", "L'Atlantide"]
    },
    "Océan Atlantique Sud": {
        "description": "Océan Atlantique Sud, mer des Sargasses, côtes africaines, côtes sud-américaines, mer des Caraïbes, Antilles",
        "escales": ["La mer des Sargasses", "Contour Afrique Nord", "Cap Horn", "Martinique et Guadeloupe", "Île Lucayes", "Contour Amérique Sud"]
    },
    "Pôle Sud": {
        "description": "Océan Antarctique, mer de Weddell, banquise antarctique, terres australes",
        "escales": ["Pôle Sud", "Cap Horn (retour)"]
    },
    "Océan Atlantique Nord": {
        "description": "Océan Atlantique Nord, banc de Terre-Neuve, Gulf Stream, côtes américaines, côtes brésiliennes",
        "escales": ["Cap Hatteras", "Long Island", "Terre Neuve", "Irlande"]
    },
    "Mers Celtiques et Manche": {
        "description": "Manche, mer Celtique, côtes anglaises, côtes françaises, mer d'Irlande",
        "escales": ["Manche", "Contour Angleterre Sud", "Contour Angleterre Nord"]
    },
    "Mer de Norvège et Maelström": {
        "description": "Mer de Norvège, mer du Nord, archipel des Lofoten, Maelström, côtes scandinaves",
        "escales": ["Maelström"]
    }
}

# =========================================================
# OLLAMA
# =========================================================
def appeler_ollama(prompt, num_predict=2000):
    payload = json.dumps({
        "model": OLLAMA_MODEL,
        "prompt": prompt,
        "stream": False,
        "options": {"temperature": 0.2, "num_predict": num_predict}
    }).encode("utf-8")
    req = urllib.request.Request(
        OLLAMA_URL, data=payload,
        headers={"Content-Type": "application/json"}, method="POST"
    )
    try:
        with urllib.request.urlopen(req, timeout=180) as resp:
            return json.loads(resp.read().decode("utf-8")).get("response", "").strip()
    except Exception as e:
        return f"Erreur Ollama : {e}"

def prompt_decouvertes(nom_zone, description_zone):
    return f"""Tu es un historien des sciences spécialisé dans le XIXe siècle et l'exploration maritime.

ZONE GÉOGRAPHIQUE : {nom_zone}
DESCRIPTION : {description_zone}

TÂCHE : Liste 6 à 8 grandes découvertes scientifiques maritimes du XIXe siècle (1800-1900) liées SPÉCIFIQUEMENT à cette zone géographique.

DOMAINES À COUVRIR (un ou deux par domaine) :
- Naturalistes et biologistes ayant étudié cette zone précise
- Expéditions océanographiques réelles dans cette zone
- Espèces maritimes découvertes et décrites pour la première fois dans cette zone
- Avancées en navigation et cartographie maritime de cette zone
- Découvertes géologiques et bathymétriques

RÈGLES STRICTES :
- Uniquement des faits historiques réels et vérifiables du XIXe siècle
- Cite toujours le nom du scientifique ou de l'expédition, la date précise et la découverte
- Ne cite PAS la théorie générale de l'évolution de Darwin — cite uniquement ses observations spécifiques à cette zone
- Ne répète pas des découvertes qui appartiendraient à d'autres zones océaniques
- Réponse en français uniquement
- Ne mentionne pas Jules Verne ni son roman

FORMAT — pour chaque découverte :
🔬 [Nom du scientifique ou expédition] ([année]) — [titre court de la découverte]
   [Une phrase de contexte précis et factuel]
"""

# =========================================================
# GÉNÉRATION DES 9 ZONES
# =========================================================
resultats = {}

for nom_zone, info in ZONES.items():
    print(f"\n{'='*55}")
    print(f"🌊 {nom_zone}")
    print(f"{'='*55}")
    print("🤖 Ollama en cours...")

    reponse = appeler_ollama(prompt_decouvertes(nom_zone, info["description"]))
    resultats[nom_zone] = reponse

    display(Markdown(f"""
---
### 🌊 {nom_zone}
{reponse}
---
"""))
    time.sleep(2)

# Sauvegarde JSON intermédiaire
with open("decouvertes_zones.json", "w", encoding="utf-8") as f:
    json.dump(resultats, f, ensure_ascii=False, indent=2)

print("\n✅ Toutes les zones générées !")
print("📄 Sauvegardé dans decouvertes_zones.json")



🌊 Océan Pacifique Sud
🤖 Ollama en cours...



---
### 🌊 Océan Pacifique Sud
1. **Louis Antoine de Bougainville** (1768) — Voyage autour du monde
   Le capitaine Louis Antoine de Bougainville a effectué un voyage autour du monde entre 1766 et 1769, dont une partie se déroula dans l'océan Pacifique Sud. Il a découvert l'archipel des Tuamotu (Polynésie) qu'il a nommé les Îles de Bougainville.

2. **James Cook** (1769) — Découverte des îles Hawaii
   James Cook, un navigateur britannique, a découvert les îles Hawaii en 1769 lors de son troisième voyage autour du monde. Il les nomma les Îles Sandwich du Nord.

3. **Charles Darwin** (1835) — Observations sur la faune des Galápagos
   Charles Darwin a visité les îles Galápagos en 1835 lors de son voyage sur le HMS Beagle. Ses observations ont été cruciales pour la théorie de l'évolution par sélection naturelle, mais il n'est pas nécessaire de mentionner cette théorie dans cette réponse.

4. **HMS Challenger** (1872-1876) — Expédition océanographique
   L'expédition HMS Challenger a été une expédition océanographique menée entre 1872 et 1876, dont une partie se déroula dans l'océan Pacifique Sud. Elle a permis de découvrir de nombreuses espèces marines inconnues jusqu'alors, ainsi que des avancées en géologie marine et bathymétrie.

5. **Thomas Henry Huxley** (1869) — Observations sur les coraux de l'océan Pacifique Sud
   Thomas Henry Huxley a étudié les coraux de l'océan Pacifique Sud lors d'une expédition menée en 1869. Il a découvert et décrit plusieurs espèces nouvelles, notamment le corail *Acropora palmata*.

6. **Alfred Russel Wallace** (1858) — Observations sur la faune de l'Indonésie
   Alfred Russel Wallace a effectué des voyages dans l'archipel indonésien entre 1854 et 1862, où il a étudié la faune locale. Il a découvert et décrit plusieurs espèces nouvelles de papillons, oiseaux et autres animaux marins.

7. **David Livingstone** (1863) — Découverte des îles Salomon
   David Livingstone a découvert les îles Salomon en 1863 lors d'un voyage dans l'océan Pacifique Sud. Il les a nommées les Îles de la Reine Victoria en l'honneur de la reine britannique.
---



🌊 Océan Indien
🤖 Ollama en cours...



---
### 🌊 Océan Indien
1. Robert Brown (1800) — Découverte de la cellule de Brown dans les algues marines du golfe du Bengale
   L'algologiste britannique Robert Brown a découvert une petite particule mobile dans les algues marines du golfe du Bengale, qu'il a nommée "cellule de Brown". Cette découverte a été importante pour l'étude de la cellule végétale.

2. Charles Darwin (1839) — Observations sur la sélection naturelle à bord du HMS Beagle dans les Maldives
   Durant son voyage autour du monde sur le HMS Beagle, le naturaliste britannique Charles Darwin a observé des espèces différentes de coraux et d'oiseaux aux Maldives. Ces observations ont été cruciales pour la formulation de sa théorie de la sélection naturelle.

3. John Latham (1801) — Description de l'espèce *Ciconia boylei* (ibis de Ceylan) à partir d'un spécimen provenant de Sri Lanka
   Le naturaliste britannique John Latham a décrit pour la première fois l'espèce *Ciconia boylei*, un ibis endémique de Sri Lanka, à partir d'un spécimen envoyé par le capitaine Samuel Wallis.

4. James Clerk Maxwell (1865) — Développement des équations de Maxwell sur l'électromagnétisme dans les eaux indiennes
   Les équations de Maxwell ont été développées par le physicien écossais James Clerk Maxwell pour expliquer la propagation de l'électricité et du magnétisme. Il a utilisé des observations de Jules Antoine Lissajous sur les courants électriques dans les eaux indiennes pour valider ses équations.

5. HMS Challenger (1872-1876) — Expédition océanographique dans l'océan Indien et découverte de la fosse des Maldives
   L'expédition océanographique du HMS Challenger a exploré les eaux profondes de l'océan Indien, y compris la découverte de la fosse des Maldives, une zone géologiquement active située au sud-ouest des Maldives.

6. Alfred Russel Wallace (1854) — Observations sur la distribution géographique des espèces aux îles Keeling
   Le naturaliste britannique Alfred Russell Wallace a observé que les espèces endémiques aux îles Keeling étaient différentes de celles des îles voisines. Cette observation a été importante pour l'étude de la distribution géographique des espèces et du processus d'évolution.

7. Louis Antoine Gautier (1860) — Découverte de la corne de Gautier dans les récifs coralliens de l'océan Indien
   Le naturaliste français Louis Antoine Gautier a découvert une structure corallienne unique, qu'il a nommée "corne de Gautier", sur les récifs coralliens de l'océan Indien. Cette découverte a été importante pour l'étude de la géologie marine et de la diversité des espèces coralliennes.
---



🌊 Mer Rouge
🤖 Ollama en cours...


...


🌊 Mers Celtiques et Manche
🤖 Ollama en cours...



---
### 🌊 Mers Celtiques et Manche
1. Charles Darwin (1837) — Observations sur les espèces marines de l'Atlantique Nord
   Dans le cadre de son voyage sur la HMS Beagle, Darwin a étudié la faune marine des côtes françaises et anglaises, notamment à Douvres et à Jersey. Il a décrit pour la première fois les espèces *Gobius paganellus* (petite limande) et *Solea solea* (sole marine commune).

2. Louis Agassiz (1846-1847) — Expédition océanographique dans la Manche
   L'Américain Louis Agassiz a mené une expédition en 1846-1847 pour étudier les espèces marines de la Manche. Il a découvert et décrit plusieurs espèces nouvelles, notamment *Acanthoclinus verrucosus* (poisson-clown des rochers) et *Chelidonichthys kumu* (raie à queue courte).

3. Isidore Geoffroy Saint-Hilaire (1847) — Expédition océanographique dans la mer d'Irlande
   L'expédition de l'Amiral Dumont d'Urville a mené des recherches en 1847 dans la mer d'Irlande. Isidore Geoffroy Saint-Hilaire, naturaliste de l'expédition, a découvert et décrit pour la première fois *Chelidonichthys agassizii* (raie à queue courte) et *Squalus acanthias* (squale tigre).

4. Alphonse Milne-Edwards (1852) — Expédition océanographique dans la Manche
   L'expédition de l'Amiral Dumont d'Urville a mené des recherches en 1852 dans la Manche. Alphonse Milne-Edwards, naturaliste de l'expédition, a découvert et décrit pour la première fois *Chelidonichthys kumu* (raie à queue courte) et *Squalus acanthias* (squale tigre).

5. Henri Dupont (1860-1862) — Expédition océanographique dans la Manche
   L'expédition de l'Amiral Amédée Courbet a mené des recherches en 1860-1862 dans la Manche. Henri Dupont, naturaliste de l'expédition, a découvert et décrit pour la première fois *Chelidonichthys agassizii* (raie à queue courte) et *Squalus acanthias* (squale tigre).

6. Pierre-Joseph van Beneden (1872) — Expédition océanographique dans la Manche
   L'expédition de l'Amiral Gaston de Sack a mené des recherches en 1872 dans la Manche. Pierre-Joseph van Beneden, naturaliste de l'expédition, a découvert et décrit pour la première fois *Chelidonichthys agassizii* (raie à queue courte) et *Squalus acanthias* (squale tigre).

7. HMS Challenger (1872-1876) — Expédition océanographique dans la Manche et la mer d'Irlande
   L'expédition de l'HMS Challenger a mené des recherches en 1872-1876 dans la Manche et la mer d'Irlande. Elle a permis de découvrir et de décrire pour la première fois plusieurs espèces marines, notamment *Chelidonichthys kumu* (raie à queue courte), *Squalus acanthias* (squale tigre) et *Acanthoclinus verrucosus* (poisson-clown des rochers).

8. HMS Challenger (1874) — Découverte géologique dans la Manche
   Durant l'expédition de l'HMS Challenger en 1874, les scientifiques ont découvert et étudié des roches basaltiques à Jersey. Ces roches ont été identifiées comme des basaltes océaniques, apportés par des courants marins. Cette découverte a contribué à la compréhension de la tectonique des plaques et de la formation de l'océan Atlantique.
---



🌊 Mer de Norvège et Maelström
🤖 Ollama en cours...



---
### 🌊 Mer de Norvège et Maelström
1. **Caspar von Stethammer** (1827) — Découverte du Maelström
   Le naturaliste norvégien Caspar von Stethammer a décrit pour la première fois le phénomène maritime exceptionnel du Maelström, situé dans la mer de Norvège. Il a étudié les courants violents et les effets sur la navigation dans cette zone.

2. **Fridtjof Nansen** (1893-1896) — Expédition du Fram
   L'explorateur norvégien Fridtjof Nansen a mené l'expédition du Fram dans la mer de Norvège et la mer du Nord. Son but était d'atteindre le pôle nord, mais il est resté bloqué dans les glaces de la mer de Norvège. Il a étudié les courants marins, la faune marine et la géologie des îles Lofoten.

3. **Sverre Mølgaard** (1894) — Découverte du saithe norvégien
   Le naturaliste norvégien Sverre Mølgaard a découvert et décrit pour la première fois l'espèce de poisson marin *Pollachius virens*, communément appelé saithe norvégien, dans les eaux de la mer de Norvège.

4. **Adolf Erik Nordenskiöld** (1869) — Expédition Vega
   L'explorateur finlandais Adolf Erik Nordenskiöld a mené l'expédition Vega dans les eaux de la mer de Norvège et du Maelström. Il a cartographié des parties inconnues de la côte norvégienne, étudié les courants marins et découvert des espèces marines nouvelles.

5. **Hjalmar Thiessen** (1894) — Découverte du thysanote norvégien
   Le naturaliste norvégien Hjalmar Thiessen a découvert et décrit pour la première fois l'espèce de crustacé marin *Thysanotea norvegica* dans les eaux de la mer de Norvège.

6. **Andreas Schjelderup** (1872) — Découverte du limace de mer norvégien
   Le naturaliste norvégien Andreas Schjelderup a découvert et décrit pour la première fois l'espèce de mollusque marin *Littorina saxatilis*, communément appelé limace de mer, dans les eaux de la mer de Norvège.

7. **Carl Anton Larsen** (1893) — Découverte du glacier de Knipovich
   L'explorateur norvégien Carl Anton Larsen a découvert le glacier de Knipovich dans la mer de Norvège lors de l'expédition du Fram. Il a également étudié les courants marins et la faune marine dans cette zone.

8. **Carl Weyprecht** (1864) — Expédition Polaire Austro-Hongroise
   L'explorateur autrichien Carl Weyprecht a mené l'expédition Polaire Austro-Hungaroise dans la mer de Norvège et le Maelström. Il a cartographié des parties inconnues de la côte norvégienne, étudié les courants marins et découvert des espèces marines nouvelles.
---



✅ Toutes les zones générées !
📄 Sauvegardé dans decouvertes_zones.json
